# Practice - 13 - Managed ingestion design and capstone assessment

Prerequisites / execution context: Core notebooks 00-12. Managed connectors and external clients additionally require source access, network connectivity, connector-specific permissions and supported workspace features.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R13: Lakeflow Connect concepts](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/); [R14: Managed database connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/cdc-overview); [R15: Query-based connectors](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/query-based-overview); [R16: Spark data sources](https://docs.databricks.com/aws/en/connect/spark-data-sources); [R17: Secret management](https://docs.databricks.com/aws/en/security/secrets/); [R70: Lakehouse Federation](https://docs.databricks.com/aws/en/query-federation/).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Written / design task 1

Choose ingestion methods for append-only cloud JSON, a supported database requiring delete capture, SaaS exports and live Teradata queries. Explain tradeoffs and source-specific prerequisites.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgMS4gQ2hvb3NlIGFuIGluZ2VzdGlvbiBhcHByb2FjaApEZXNpZ24gZm91ciB3b3JrbG9hZHM6IGFwcGVuZC1vbmx5IGNsb3VkIEpTT047IFNRTCBTZXJ2ZXIgd2l0aCByZXF1aXJlZCBkZWxldGVzOyBkYWlseSBTYWFTIGV4cG9ydHM7IGFkIGhvYyBsaXZlIFRlcmFkYXRhIHF1ZXJpZXMuIFN1Z2dlc3RlZCBzdGFydGluZyBwb2ludHM6IEF1dG8gTG9hZGVyOyBhIHN1cHBvcnRlZCBtYW5hZ2VkIENEQyBjb25uZWN0b3I7IGEgc3VwcG9ydGVkIG1hbmFnZWQgU2FhUyBjb25uZWN0b3Igb3IgZmlsZSBpbmdlc3Rpb247IExha2Vob3VzZSBGZWRlcmF0aW9uLiBDb25maXJtIHRoZSBzb3VyY2Utc3BlY2lmaWMgY29ubmVjdG9yIHN1cHBvcnQgYW5kIHJlbGVhc2Ugc3RhdGUgaW5zdGVhZCBvZiBhc3N1bWluZyBhbGwgZGF0YWJhc2VzIGFyZSBpbnRlcmNoYW5nZWFibGUuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9pbmdlc3Rpb24vY2xvdWQtb2JqZWN0LXN0b3JhZ2UvYXV0by1sb2FkZXIiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTA6IEF1dG8gTG9hZGVyPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9pbmdlc3Rpb24vbGFrZWZsb3ctY29ubmVjdC8iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTM6IExha2VmbG93IENvbm5lY3QgY29uY2VwdHM8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2luZ2VzdGlvbi9sYWtlZmxvdy1jb25uZWN0L2NkYy1vdmVydmlldyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIxNDogTWFuYWdlZCBkYXRhYmFzZSBjb25uZWN0b3JzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9xdWVyeS1mZWRlcmF0aW9uLyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI3MDogTGFrZWhvdXNlIEZlZGVyYXRpb248L2E+Lgo8L2Rpdj48L2RldGFpbHM+')


## Written / design task 2

Plan initial and incremental managed connector ingestion, including a source update/delete test where supported, monitoring, credentials and recovery. Distinguish a design exercise from a real connector run.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDIgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgMi4gTWFuYWdlZCBjb25uZWN0b3Igd2Fsa3Rocm91Z2gKV2l0aCBhbiBhcHByb3ZlZCBkaXNwb3NhYmxlIHNvdXJjZSwgc2VsZWN0IGEgc3VwcG9ydGVkIGNvbm5lY3RvciBpbiB0aGUgaW5nZXN0aW9uIFVJLiBGb2xsb3cgaXRzIGxpbmtlZCBzb3VyY2Utc2V0dXAgaW5zdHJ1Y3Rpb25zLiBDcmVhdGUvdXNlIHRoZSBVQyBjb25uZWN0aW9uOyBjaG9vc2Ugc291cmNlIG9iamVjdHMgYW5kIFVDIGRlc3RpbmF0aW9uOyBjb25maWd1cmUgdGhlIGdhdGV3YXkvc3RhZ2luZyBjb21wb25lbnRzIGlmIHJlcXVpcmVkIGJ5IHRoYXQgY29ubmVjdG9yOyBydW4gaW5pdGlhbCBpbmdlc3Rpb24sIHRoZW4gYW4gdXBkYXRlIGFuZCBkZWxldGUgaWYgQ0RDIGlzIHN1cHBvcnRlZC4gQ29tcGFyZSBpbml0aWFsIHJvd3MgYW5kIHN1YnNlcXVlbnQgdGFyZ2V0IGNoYW5nZXMuIEluc3BlY3QgcGlwZWxpbmUgc3RhdHVzLCBwZXJtaXNzaW9ucywgYW5kIHJlZnJlc2ggY2FkZW5jZS4gUmVjb3JkIHNvdXJjZSBsb2FkIGFuZCByZWNvdmVyeSBiZWhhdmlvci4gSWYgbm8gZXh0ZXJuYWwgc291cmNlIGlzIGF2YWlsYWJsZSwgZG9jdW1lbnQgdGhlc2Ugc3RlcHMgYW5kIHRoZSByZXF1aXJlZCBhY2Nlc3M7IHRoZSBsb2NhbCBzeW50aGV0aWMgbGFicyBkbyBub3Qgc2ltdWxhdGUgbWFuYWdlZCBpbmdlc3Rpb24uIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9pbmdlc3Rpb24vbGFrZWZsb3ctY29ubmVjdC8iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTM6IExha2VmbG93IENvbm5lY3QgY29uY2VwdHM8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2luZ2VzdGlvbi9sYWtlZmxvdy1jb25uZWN0L2NkYy1vdmVydmlldyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIxNDogTWFuYWdlZCBkYXRhYmFzZSBjb25uZWN0b3JzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9pbmdlc3Rpb24vbGFrZWZsb3ctY29ubmVjdC9xdWVyeS1iYXNlZC1vdmVydmlldyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIxNTogUXVlcnktYmFzZWQgY29ubmVjdG9yczwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')


## Coding task 1: Land an offline REST-response fixture

Two pages contain item a/value10 and item b/value20. Write each page as an immutable JSON-lines file under a fresh volume path; read with explicit id STRING,value LONG schema as landed. Explain what this fixture cannot test about a live API.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
import json, uuid
pages = [{"items":[{"id":"a","value":10}],"next":"page2"},
         {"items":[{"id":"b","value":20}],"next":None}]

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert landed.count() == 2
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIEpEQkMvT0RCQy9SRVNUIGRlc2lnbiBleGVyY2lzZQpVc2Ugc3VwcG9ydGVkIFNwYXJrIGRhdGEgc291cmNlcyBvciBhIFVDIGNvbm5lY3Rpb24gZm9yIEpEQkMgaW5nZXN0aW9uOyB2ZXJpZnkgZHJpdmVyL25ldHdvcmsgc3VwcG9ydCBhbmQgY29udHJvbCByZWFkIHBhcmFsbGVsaXNtIHRvIGF2b2lkIG92ZXJ3aGVsbWluZyB0aGUgc291cmNlLiBPREJDL0pEQkMgY2xpZW50cyBjb25uZWN0aW5nIFRPIERhdGFicmlja3MgZm9yIHF1ZXJpZXMgZGlmZmVyIGZyb20gU3BhcmsgcmVhZGluZyBGUk9NIGEgZGF0YWJhc2UuIEEgY3VzdG9tIFJFU1QgY2xpZW50IG5lZWRzIHBhZ2luYXRpb24sIHJldHJ5L2JhY2tvZmYsIHJhdGUtbGltaXQgaGFuZGxpbmcsIGEgZHVyYWJsZSBleHRyYWN0aW9uIGN1cnNvciwgc2VjcmV0cywgYW5kIGEgcmVwbGF5LXNhZmUgbGFuZGluZyBjb250cmFjdC4gVGhlIG5leHQgY2VsbCB1c2VzIGFuIG9mZmxpbmUgcmVzcG9uc2UgZml4dHVyZSB0byBwcmFjdGljZSBsYW5kaW5nOyBpdCBkb2VzIG5vdCBjbGFpbSB0byB0ZXN0IGEgbGl2ZSBzZXJ2aWNlLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vY29ubmVjdC9zcGFyay1kYXRhLXNvdXJjZXMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTY6IFNwYXJrIGRhdGEgc291cmNlczwvYT47IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vc2VjdXJpdHkvc2VjcmV0cy8iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTc6IFNlY3JldCBtYW5hZ2VtZW50PC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmltcG9ydCBqc29uLHV1aWQKIyBBIGRldGVybWluaXN0aWMgc3RhbmQtaW4gZm9yIHR3byBwYWdlcyByZXR1cm5lZCBieSBhbiBhdXRob3JpemVkIEFQSSBjbGllbnQuCnBhZ2VzPVt7JnF1b3Q7aXRlbXMmcXVvdDs6W3smcXVvdDtpZCZxdW90OzomcXVvdDthJnF1b3Q7LCZxdW90O3ZhbHVlJnF1b3Q7OjEwfV0sJnF1b3Q7bmV4dCZxdW90OzomcXVvdDtwYWdlMiZxdW90O30sCiAgICAgICB7JnF1b3Q7aXRlbXMmcXVvdDs6W3smcXVvdDtpZCZxdW90OzomcXVvdDtiJnF1b3Q7LCZxdW90O3ZhbHVlJnF1b3Q7OjIwfV0sJnF1b3Q7bmV4dCZxdW90OzpOb25lfV0KcnVuX2lkPXV1aWQudXVpZDQoKS5oZXhbOjEwXQpwYXRoPWYmcXVvdDt7Vk9MVU1FfS9yZXN0X2ZpeHR1cmUve3J1bl9pZH0mcXVvdDsKZm9yIG4scGFnZSBpbiBlbnVtZXJhdGUocGFnZXMpOgogICAgZGJ1dGlscy5mcy5wdXQoZiZxdW90O3twYXRofS9wYWdlX3tufS5qc29uJnF1b3Q7LCZxdW90O1xuJnF1b3Q7LmpvaW4oanNvbi5kdW1wcyh4KSBmb3IgeCBpbiBwYWdlWyZxdW90O2l0ZW1zJnF1b3Q7XSksVHJ1ZSkKbGFuZGVkPXNwYXJrLnJlYWQuc2NoZW1hKCZxdW90O2lkIFNUUklORyx2YWx1ZSBMT05HJnF1b3Q7KS5qc29uKHBhdGgpCmFzc2VydCBsYW5kZWQuY291bnQoKT09Mgo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 2: Optional: ingest through a governed JDBC connection

Use your approved JDBC UC connection and disposable source query to read id,name and persist BRONZE.jdbc_snapshot. Keep execution opt-in; do not hardcode credentials. Explain why this snapshot is not automatically CDC.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
RUN_JDBC = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIyBPcHRpb25hbCBKREJDIGluZ2VzdGlvbiB3aXRoIGEgZ292ZXJuZWQgY29ubmVjdGlvbgpSZXF1aXJlcyBhbiBhZG1pbmlzdHJhdG9yLWNyZWF0ZWQgSkRCQyBVQyBjb25uZWN0aW9uIGFuZCBhbiBhcHByb3ZlZCBkaXNwb3NhYmxlIHNvdXJjZSB0YWJsZS4gRWRpdCB0aGUgY29ubmVjdGlvbiBuYW1lIGFuZCBxdWVyeSBmb3IgeW91ciBzb3VyY2UuIFRoaXMgY2VsbCBjb3BpZXMgcm93cyBpbnRvIGEgbWFuYWdlZCBEZWx0YSB0YWJsZTsgaXQgaXMgbm90IGZlZGVyYXRpb24gYWxvbmUuIEl0IGlzIGRpc2FibGVkIGJ5IGRlZmF1bHQuIDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vY29ubmVjdC9zcGFyay1kYXRhLXNvdXJjZXMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5PZmZpY2lhbCBTcGFyayBkYXRhIHNvdXJjZXM8L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlJVTl9KREJDPUZhbHNlCmlmIFJVTl9KREJDOgogICAgQ09OTkVDVElPTj0mcXVvdDthcHByb3ZlZF9sYWJfamRiY19jb25uZWN0aW9uJnF1b3Q7CiAgICAjIEFkYXB0IHF1b3RlZCBpZGVudGlmaWVycyBhbmQgY29sdW1uIG5hbWVzIHRvIHRoZSBzb3VyY2UgZGF0YWJhc2UuCiAgICBRVUVSWT0mcXVvdDtTRUxFQ1QgaWQsIG5hbWUgRlJPTSBhcHByb3ZlZF9sYWJfc291cmNlJnF1b3Q7CiAgICBleHRyYWN0ZWQ9KHNwYXJrLnJlYWQuZm9ybWF0KCZxdW90O2pkYmMmcXVvdDspCiAgICAgICAgICAgICAgIC5vcHRpb24oJnF1b3Q7ZGF0YWJyaWNrcy5jb25uZWN0aW9uJnF1b3Q7LENPTk5FQ1RJT04pCiAgICAgICAgICAgICAgIC5vcHRpb24oJnF1b3Q7cXVlcnkmcXVvdDssUVVFUlkpLmxvYWQoKSkKICAgIGFzc2VydCB7JnF1b3Q7aWQmcXVvdDssJnF1b3Q7bmFtZSZxdW90O30gJmx0Oz0gc2V0KGV4dHJhY3RlZC5jb2x1bW5zKQogICAgVEFSR0VUPWYmcXVvdDt7QlJPTlpFfS5qZGJjX3NuYXBzaG90JnF1b3Q7CiAgICBleHRyYWN0ZWQud3JpdGUuZm9ybWF0KCZxdW90O2RlbHRhJnF1b3Q7KS5tb2RlKCZxdW90O292ZXJ3cml0ZSZxdW90Oykuc2F2ZUFzVGFibGUoVEFSR0VUKQogICAgc3BhcmsudGFibGUoVEFSR0VUKS5zaG93KDEwLHRydW5jYXRlPUZhbHNlKQogICAgIyBBIGxhdGVyIGluY3JlbWVudGFsIHZlcnNpb24gbmVlZHMgYSBkdXJhYmxlIGN1cnNvci9DREMgY29udHJhY3Q7CiAgICAjIHJlcGVhdGVkIHNuYXBzaG90cyBhcmUgbm90IGFuIHVwZGF0ZS9kZWxldGUtYXdhcmUgcmVwbGljYXRpb24gZGVzaWduLgo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 3: Optional: query Databricks from a local ODBC client

On a separately configured local machine, use an authenticated DSN named in DATABRICKS_LAB_DSN. Validate the non-secret DSN name, open a connection/cursor and query current_catalog/current_schema. Keep disabled in normal Databricks execution.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
RUN_LOCAL_ODBC = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIyBPcHRpb25hbCBsb2NhbCBPREJDIGNsaWVudApSdW4gdGhpcyBvbiBhIGxvY2FsIGRldmVsb3BtZW50IG1hY2hpbmUgd2l0aCB0aGUgRGF0YWJyaWNrcyBPREJDIGRyaXZlciBhbmQgcHlvZGJjIGluc3RhbGxlZCBhbmQgYW4gYXV0aGVudGljYXRlZCBEU04gY29uZmlndXJlZCwgZm9sbG93aW5nIHRoZSBvZmZpY2lhbCBzZXR1cC4gVGhpcyBkaXJlY3Rpb24gcXVlcmllcyBEYXRhYnJpY2tzIEZST00gYSBjbGllbnQuIFVzZSB5b3VyIGRyaXZlciYjeDI3O3MgYXBwcm92ZWQgT0F1dGggc2V0dXA7IGRvIG5vdCBwYXN0ZSB0b2tlbnMgaW50byB0aGlzIG5vdGVib29rLiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2Rldi10b29scy9weW9kYmMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5PZmZpY2lhbCBweW9kYmMgdHV0b3JpYWw8L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlJVTl9MT0NBTF9PREJDPUZhbHNlCmlmIFJVTl9MT0NBTF9PREJDOgogICAgaW1wb3J0IG9zLHJlLHB5b2RiYwogICAgIyBEU04gaXMgYSBjb25maWd1cmVkIG5vbi1zZWNyZXQgbmFtZSwgbm90IGEgcGFzc3dvcmQuCiAgICBkc249b3MuZW52aXJvblsmcXVvdDtEQVRBQlJJQ0tTX0xBQl9EU04mcXVvdDtdCiAgICBhc3NlcnQgcmUuZnVsbG1hdGNoKHImcXVvdDtbQS1aYS16MC05Xy1dKyZxdW90Oyxkc24pCiAgICB3aXRoIHB5b2RiYy5jb25uZWN0KGYmcXVvdDtEU049e2Rzbn0mcXVvdDssYXV0b2NvbW1pdD1UcnVlKSBhcyBjb25uZWN0aW9uOgogICAgICAgIHdpdGggY29ubmVjdGlvbi5jdXJzb3IoKSBhcyBjdXJzb3I6CiAgICAgICAgICAgIGN1cnNvci5leGVjdXRlKCZxdW90O1NFTEVDVCBjdXJyZW50X2NhdGFsb2coKSwgY3VycmVudF9zY2hlbWEoKSZxdW90OykKICAgICAgICAgICAgcHJpbnQoY3Vyc29yLmZldGNob25lKCkpCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Written / design task 3

Capstone: rebuild ingestion, quarantine, latest orders, customer enrichment and Gold revenue from the fixture. Prove rerun behavior, both Job branches, environment isolation and monitoring. State row grain and expected counts BEFORE revealing the acceptance contract.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDMgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNC4gQ2Fwc3RvbmU6IHJlYnVpbGQgZnJvbSByZXF1aXJlbWVudHMKV2l0aG91dCByZWFkaW5nIHNvbHV0aW9uczogaW5nZXN0IGVpZ2h0IHNlZWQgZXZlbnRzOyBxdWFyYW50aW5lIHR3byBpbnZhbGlkIGFtb3VudHM7IHByb2R1Y2UgZm91ciBsYXRlc3QgdmFsaWQgb3JkZXJzOyByZXRhaW4gdHdvIHVubWF0Y2hlZCBjdXN0b21lcnMgd2l0aCBhIGxlZnQgam9pbjsgcHVibGlzaCB0b3RhbCByZXZlbnVlIDIyMC4wMDsgcHJvdmUgcmVydW4gYmVoYXZpb3I7IGNob29zZSBhIGRhdGEtZHJpdmVuIHRyaWdnZXI7IGRlbW9uc3RyYXRlIGJvdGggam9iIGJyYW5jaGVzOyBkZXBsb3kgZGV2L3Rlc3QgdGFyZ2V0czsgZG9jdW1lbnQgcmVhZGVyIGFjY2VzcyBhbmQgbWFza2VkIGZpZWxkczsgZGlhZ25vc2UgYSBwcm92aWRlZCBza2V3IHNjZW5hcmlvLiBTdWJtaXQgcnVuIElEcywgYXNzZXJ0aW9ucywgYSBncmFpbiBkZWZpbml0aW9uLCBsaW5lYWdlIGV2aWRlbmNlLCBhbmQgYSByZWNvdmVyeSBwbGFuLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vbGFrZWhvdXNlL21lZGFsbGlvbiIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIwNjogTWVkYWxsaW9uIGFyY2hpdGVjdHVyZTwvYT47IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vam9icy8iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMzc6IExha2VmbG93IEpvYnM8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2Rldi10b29scy9idW5kbGVzLyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI0NjogRGVjbGFyYXRpdmUgQXV0b21hdGlvbiBCdW5kbGVzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9kYXRhLWdvdmVybmFuY2UvdW5pdHktY2F0YWxvZy9hY2Nlc3MtY29udHJvbC9wcml2aWxlZ2VzLXJlZmVyZW5jZSIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI1ODogVUMgcHJpdmlsZWdlcyByZWZlcmVuY2U8L2E+Lgo8L2Rpdj48L2RldGFpbHM+')


## Written / design task 4

Design a readiness rubric based on correctness, explanation, replay, diagnosis, permissions, isolation, observability and recovery; score your evidence, not memorization.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDQgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgRXZhbHVhdGlvbiBydWJyaWMgKGF1dGhvci1kZXNpZ25lZCkKU2NvcmUgMC8xLzIgZm9yIGVhY2g6IGNvcnJlY3QgcmVzdWx0LCBleHBsYW5hdGlvbiwgcmVydW4gcHJvb2YsIGZhaWx1cmUgZGlhZ25vc2lzLCBwZXJtaXNzaW9uIGRlc2lnbiwgZW52aXJvbm1lbnQgc2VwYXJhdGlvbiwgb2JzZXJ2YWJpbGl0eSwgYW5kIHJlY292ZXJ5LiBBIHN1Z2dlc3RlZCBwcmFjdGljZSBnb2FsIGlzIDE0LzE2IHdpdGggbm8gemVybyBpbiBjb3JyZWN0bmVzcyBvciBzZWN1cml0eS4gVGhpcyBpcyBhIGxlYXJuaW5nIHJ1YnJpYywgbm90IGFuIG9mZmljaWFsIHBhc3Npbmcgc2NvcmUuIFJldmlldyBTVFVEWV9HVUlERS5tZCBmb3IgdGhlIG9iamVjdGl2ZSBtYXRyaXggYW5kIHNjZW5hcmlvIGFuc3dlciBrZXkuCjwvZGl2PjwvZGV0YWlscz4=')
